# Modeles IA - Prediction des Carences en Vitamines
## Projet Vitamin_IA

Ce notebook est la suite de data_vis.ipynb.
L'exploration des donnees a montre un desequilibre important des classes (1509 Healthy vs 95 Scurvy).
On applique ici : Preprocessin -> SMOTE -> Modeles -> Evaluation -> Comparaison

Modeles testes : Random Forest, SVM, KNN

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, ConfusionMatrixDisplay)
import pickle

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print("SMOTE disponible")
except ImportError:
    SMOTE_AVAILABLE = False
    print("SMOTE non disponible - pip install imbalanced-learn")

print("Imports OK")

## 2. Chargement des Donnees

Meme dataset que data_vis.ipynb

In [ ]:
import os

possible_paths = [
    "../data_csv/raw/vitamin_deficiency_disease_dataset_20260123.ods",
    "../data_csv_equilibree/vitamin_balanced_50.csv",
    "../bdds_pour_tests/paire1_vitamines_top_symptoms_95.csv",
]

df = None
for path in possible_paths:
    if os.path.exists(path):
        if path.endswith('.ods'):
            df = pd.read_excel(path, engine='odf')
        else:
            df = pd.read_csv(path)
        print(f"Donnees chargees : {path}")
        break

if df is None:
    print("Fichier non trouve - verifiez le chemin")
else:
    print(f"{df.shape[0]} lignes x {df.shape[1]} colonnes")
    print("\nDistribution des classes :")
    print(df['disease_diagnosis'].value_counts())

## 3. Preprocessing

Les modeles ML ne comprennent que des chiffres.
Il faut encoder les variables texte et normaliser les valeurs numeriques.

In [ ]:
df_proc = df.copy()

# Supprimer colonnes non pertinentes
cols_to_drop = [c for c in ['symptoms_list', 'latitude_region', 'income_level'] if c in df_proc.columns]
df_proc = df_proc.drop(columns=cols_to_drop)
print(f"Colonnes supprimees : {cols_to_drop}")

# Encoder variables categorielles
cat_cols = df_proc.select_dtypes(include=['object']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'disease_diagnosis']

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))
    le_dict[col] = le

print(f"\nColonnes encodees : {cat_cols}")

# Encoder la cible
le_target = LabelEncoder()
df_proc['target'] = le_target.fit_transform(df_proc['disease_diagnosis'])

print("\nClasses :")
for i, c in enumerate(le_target.classes_):
    print(f"  {i} -> {c}")

# Features / cible
feature_cols = [c for c in df_proc.columns if c not in ['disease_diagnosis', 'target']]
X = df_proc[feature_cols]
y = df_proc['target']
print(f"\nX : {X.shape} | y : {y.shape}")

In [ ]:
# Split train/test stratifie (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisation StandardScaler (indispensable pour SVM et KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train : {X_train.shape[0]} echantillons")
print(f"Test  : {X_test.shape[0]} echantillons")
print("Normalisation OK (StandardScaler)")

## 4. SMOTE - Reequilibrage des Classes

### Pourquoi SMOTE ?
L'exploration dans data_vis.ipynb a revele un fort desequilibre :

| Maladie | Effectif |
|---------|----------|
| Healthy | 1509 |
| Anemia | 1245 |
| Rickets_Osteomalacia | 1029 |
| Night_Blindness | 122 |
| Scurvy | 95 |

- Ratio 15:1 entre Healthy et Scurvy
- Sans correction, le modele ignore Scurvy et Night_Blindness
- SMOTE genere des exemples synthetiques par interpolation entre voisins proches

In [ ]:
if SMOTE_AVAILABLE:
    smote = SMOTE(random_state=42, k_neighbors=3)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    labels = le_target.classes_

    counts_before = pd.Series(y_train).value_counts().sort_index()
    counts_after  = pd.Series(y_train_smote).value_counts().sort_index()

    axes[0].bar(labels, [counts_before.get(i,0) for i in range(len(labels))], color='#FF5722', alpha=0.85)
    axes[0].set_title("Avant SMOTE", fontweight='bold', fontsize=12)
    axes[0].tick_params(axis='x', rotation=30)

    axes[1].bar(labels, [counts_after.get(i,0) for i in range(len(labels))], color='#4CAF50', alpha=0.85)
    axes[1].set_title("Apres SMOTE", fontweight='bold', fontsize=12)
    axes[1].tick_params(axis='x', rotation=30)

    plt.suptitle("Impact du SMOTE sur la distribution", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('smote_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"SMOTE : {len(y_train)} -> {len(y_train_smote)} echantillons d'entrainement")
else:
    X_train_smote, y_train_smote = X_train_scaled, y_train
    print("Entrainement sans SMOTE")

## 5. Entrainement des Modeles

### Pourquoi ces 3 modeles ?
- Random Forest : robuste, gere bien les donnees heterogenes, resistant au surapprentissage
- SVM : efficace sur des espaces de grande dimension, bonnes frontieres non-lineaires
- KNN : simple, bon baseline, sensible a la normalisation

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=15,
        min_samples_split=5, random_state=42, n_jobs=-1
    ),
    "SVM": SVC(
        kernel='rbf', C=10, gamma='scale',
        random_state=42, probability=True
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=7, weights='distance', metric='euclidean'
    )
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 60)
for name, model in models.items():
    print(f"\nEntrainement : {name} ...")
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    cv_scores = cross_val_score(model, X_train_smote, y_train_smote,
                                cv=cv, scoring='f1_weighted', n_jobs=-1)
    results[name] = {
        'model': model, 'y_pred': y_pred,
        'accuracy': acc, 'f1_weighted': f1,
        'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()
    }
    print(f"  Accuracy : {acc:.4f} | F1 : {f1:.4f} | CV : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

print("\nTous les modeles entraines !")

## 6. Evaluation des Modeles

In [ ]:
# Matrices de confusion
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=le_target.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{name}\nAcc: {res['accuracy']:.3f} | F1: {res['f1_weighted']:.3f}",
                 fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=35)

plt.suptitle("Matrices de Confusion", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Rapports detailles
for name, res in results.items():
    print(f"\n{'='*55}")
    print(f"{name}")
    print('='*55)
    print(classification_report(y_test, res['y_pred'], target_names=le_target.classes_))

## 7. Comparaison des Modeles

In [ ]:
comparison = pd.DataFrame({
    'Modele'      : list(results.keys()),
    'Accuracy'    : [r['accuracy']    for r in results.values()],
    'F1 Weighted' : [r['f1_weighted'] for r in results.values()],
    'CV Mean'     : [r['cv_mean']     for r in results.values()],
    'CV Std'      : [r['cv_std']      for r in results.values()]
}).sort_values('F1 Weighted', ascending=False).reset_index(drop=True)

display(comparison.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison))
w = 0.25

b1 = ax.bar(x - w, comparison['Accuracy'],    w, label='Accuracy',    color='#2196F3', alpha=0.85)
b2 = ax.bar(x,     comparison['F1 Weighted'], w, label='F1 Weighted', color='#4CAF50', alpha=0.85)
b3 = ax.bar(x + w, comparison['CV Mean'],     w, label='CV Mean',     color='#FF9800', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(comparison['Modele'])
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Comparaison des Performances', fontsize=13, fontweight='bold')
ax.legend()

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.3f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Importance des Features (Random Forest)

In [ ]:
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
colors_bar = plt.cm.Blues(np.linspace(0.4, 0.9, len(importances)))[::-1]
importances.plot(kind='barh', color=colors_bar)
plt.title("Top 15 Features les plus importantes", fontsize=13, fontweight='bold')
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 :")
for i, (feat, imp) in enumerate(importances.head(10).items(), 1):
    print(f"  {i:2}. {feat:<35} {imp:.4f}")

## 9. Conclusion

### Bilan methodologique

| Etape | Choix | Justification |
|-------|-------|---------------|
| Reequilibrage | SMOTE | Ratio 15:1 entre Healthy et Scurvy (vu dans data_vis.ipynb) |
| Split | 80/20 stratifie | Preserve la distribution des classes |
| Normalisation | StandardScaler | Indispensable pour SVM et KNN |
| Validation | StratifiedKFold (5) | Robustesse de l'evaluation |
| Metrique | F1 weighted | Adapte aux classes desequilibrees |

### Recommandations
- Tester XGBoost pour potentiellement ameliorer les performances
- Optimiser les hyperparametres avec GridSearchCV
- Attention au data leakage : les colonnes has_* sont directement liees aux symptomes diagnostiques

In [ ]:
# Sauvegarde du meilleur modele
best_name  = comparison.iloc[0]['Modele']
best_model = results[best_name]['model']

with open(f'best_model_{best_name.replace(" ", "_")}.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le_target, f)

print(f"Meilleur modele : {best_name}")
print(f"Accuracy : {results[best_name]['accuracy']:.4f}")
print(f"F1 Score : {results[best_name]['f1_weighted']:.4f}")
print("Sauvegardes : best_model.pkl | scaler.pkl | label_encoder.pkl")